# MammosighTR — Veri Ön İşleme ve Hazırlama Pipeline'ı

Bu çalışma, T.C. Sağlık Bakanlığı Açık Veri Portalı üzerinde sunulan **MammosighTR** mamografi veri setinin klinik sınıflandırma modelleri için hazırlanması amacıyla geliştirilmiş veri işleme pipeline'ını içermektedir.

### Metodoloji ve İşlem Basamakları:
1. **Hasta Düzeyinde Tabakalı Bölümleme:** Hasta (`case_id`) düzeyinde %70 Eğitim, %15 Doğrulama ve %15 Test tabakalı ayrımı ile veri sızıntısı (data leakage) engellenmiştir.
2. **Tıbbi Görüntü Ön İşleme:**
   - DICOM VOI LUT dönüşümü, `[p1, p99.5]` persentil pencereleme ve `MONOCHROME1` polarite düzeltmesi.
   - Morfolojik bağlı bileşen analizi ile radyo-opak işaretleyici (lead marker) ve artefakt temizliği.
   - Anatomik yön standardizasyonu (göğüs duvarının sol dikey eksene hizalanması) ve en-boy oranını koruyan dolgulu boyutlandırma (512x512 letterbox).
   - 3-Kanallı kontrast temsili: R=Global Gamma Düzeltmesi (γ=0.85), G=Adaptif Histogram Eşitleme (CLAHE), B=Yüksek Frekans Detay İyileştirme (Unsharp Masking).
3. **Akış ve Kontrol Noktası Mimarisi:** İndirme ve işleme sürecinde bellek ve arşiv bütünlüğünü koruyan periyodik kontrol noktası mekanizması.


In [ ]:
# ==============================================================================
# Bölüm 1: Ortam Konfigürasyonu, Bağımlılıklar ve Başlatma
# ==============================================================================
import os, sys, glob, shutil, zipfile, io, json, time, random, subprocess, requests, warnings, socket
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pydicom', 'openpyxl', 'beautifulsoup4', 'requests', 'tqdm', 'pandas', 'numpy', 'Pillow', 'opencv-python-headless', 'stream-unzip'], check=False)
import numpy as np, pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from PIL import Image
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
import cv2
from sklearn.model_selection import train_test_split
from stream_unzip import stream_unzip

warnings.filterwarnings("ignore")
socket.setdefaulttimeout(60)

SESSION_START_TIME = time.time()
MAX_SESSION_HOURS  = 10.5   # Kaggle 12 saat sınırından önce (10.5 saatte) güvenli kayıt yapıp durur
FLUSH_EVERY        = 200    # Her 200 görüntüde bir zip dosyasını diske flush et (veri kaybını %100 önler)

if os.path.exists('/kaggle'):
    print("Kaggle ortamı algılandı.")
    WORK_DIR      = "/kaggle/working"
    TEMP_DIR      = "/kaggle/temp/MammosighTR_Data"
    EXTRACTED_DIR = "/kaggle/temp/extracted"
elif os.path.exists('/content'):
    print("Google Colab ortamı algılandı.")
    WORK_DIR      = "/content"
    TEMP_DIR      = "/content/MammosighTR_Data"
    EXTRACTED_DIR = "/content/extracted"
else:
    print("Yerel bilgisayar ortamı algılandı.")
    WORK_DIR      = "."
    TEMP_DIR      = "./temp_mammosight"
    EXTRACTED_DIR = "./temp_extracted"

out_zip_path   = os.path.join(WORK_DIR, "yolo_dataset.zip")
ckpt_json_path = os.path.join(WORK_DIR, "prep_checkpoint.json")
tail_bin_path  = os.path.join(WORK_DIR, "prep_tail_carryover.bin")

for p in [TEMP_DIR, EXTRACTED_DIR]:
    if os.path.exists(p):
        shutil.rmtree(p, ignore_errors=True)
os.makedirs(TEMP_DIR, exist_ok=True)
os.makedirs(EXTRACTED_DIR, exist_ok=True)

# Kontrol noktası doğrulaması: Önceki çalışma çıktıları tespit edilerek çalışma alanına aktarılır
if os.path.exists('/kaggle/input'):
    if not os.path.exists(out_zip_path):
        prev_zips = glob.glob('/kaggle/input/**/yolo_dataset.zip', recursive=True)
        if prev_zips:
            print(f"Önceki oturumdan yolo_dataset.zip bulundu, kopyalanıyor: {prev_zips[0]}")
            shutil.copy2(prev_zips[0], out_zip_path)
    if not os.path.exists(ckpt_json_path):
        prev_ckpts = glob.glob('/kaggle/input/**/prep_checkpoint.json', recursive=True)
        if prev_ckpts:
            shutil.copy2(prev_ckpts[0], ckpt_json_path)
    if not os.path.exists(tail_bin_path):
        prev_tails = glob.glob('/kaggle/input/**/prep_tail_carryover.bin', recursive=True)
        if prev_tails:
            shutil.copy2(prev_tails[0], tail_bin_path)

existing_arcnames = set()
if os.path.exists(out_zip_path):
    try:
        with zipfile.ZipFile(out_zip_path, 'r') as zf_check:
            existing_arcnames = set(zf_check.namelist())
        print(f"Mevcut arşiv kontrol ediliyor: Mevcut yolo_dataset.zip içinde {len(existing_arcnames)} görüntü bulundu.")
    except Exception as e:
        print(f"Mevcut zip bozuk olduğu için sıfırlanıyor: {e}")
        os.remove(out_zip_path)

prep_state = {"completed_groups": [], "group_progress": {}}
if os.path.exists(ckpt_json_path):
    try:
        with open(ckpt_json_path, "r", encoding="utf-8") as f:
            prep_state = json.load(f)
        print(f"Kayıtlı İlerleme Yüklendi: {prep_state}")
    except Exception as e:
        print(f"prep_checkpoint.json okunamadı: {e}")

def save_prep_state():
    tmp_p = ckpt_json_path + ".tmp"
    with open(tmp_p, "w", encoding="utf-8") as f:
        json.dump(prep_state, f, indent=2, ensure_ascii=False)
    os.replace(tmp_p, ckpt_json_path)

DATASET_URL   = "https://acikveri.saglik.gov.tr/Home/DataSetDetail/3"
BASE_URL      = "https://acikveri.saglik.gov.tr"
CLASS_NAMES   = ["BIRADS0", "BIRADS1_2", "BIRADS4_5"]
DISPLAY_NAMES = ["BI-RADS 0", "BI-RADS 1-2", "BI-RADS 4-5"]
IMG_SIZE      = 512
USE_TRAIN2    = False  # MG_EGITIM_1 (18,879 vaka) için False; MG_EGITIM_2'yi de katmak için True


In [ ]:
# ==============================================================================
# Bölüm 2: Klinik Metadata Dosyasının (Bilgi.zip) Temini ve Çıkarılması
# ==============================================================================
print("Bilgi.zip indiriliyor...")
res = requests.get(DATASET_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
soup = BeautifulSoup(res.text, "html.parser")
bilgi_url = None
for a in soup.find_all("a", href=True):
    href = a["href"]
    text = a.get_text(strip=True)
    if ("Download" in href or "Document" in href) and "bilgi" in text.lower():
        bilgi_url = BASE_URL + href if href.startswith("/") else href
        break

if bilgi_url:
    bilgi_dest = os.path.join(TEMP_DIR, "Bilgi.zip")
    cmd = ["curl", "--retry", "5", "--retry-delay", "3", "-H", "User-Agent: Mozilla/5.0",
           "-L", "-f", "--connect-timeout", "30", "-m", "120", "-o", bilgi_dest, bilgi_url]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if result.returncode == 0 and os.path.exists(bilgi_dest):
        with zipfile.ZipFile(bilgi_dest, "r") as zf:
            zf.extractall(EXTRACTED_DIR)
        os.remove(bilgi_dest)
        print("Bilgi.zip indirildi ve çıkarıldı.")
    else:
        raise RuntimeError(f"Bilgi.zip indirilemedi: {result.stderr.decode()}")
else:
    raise RuntimeError("Bilgi.zip indirme bağlantısı bulunamadı!")


In [ ]:
# ==============================================================================
# Bölüm 3: Klinik Metadata Ayrıştırma ve Hasta Düzeyinde Tabakalı Bölümleme
# ==============================================================================
xlsx_files = glob.glob(os.path.join(EXTRACTED_DIR, "**", "*.xlsx"), recursive=True)
TRAIN1 = [f for f in xlsx_files if "TRAIN1" in f.upper()][0] if any("TRAIN1" in f.upper() for f in xlsx_files) else ""
TRAIN2 = [f for f in xlsx_files if "TRAIN2" in f.upper()][0] if any("TRAIN2" in f.upper() for f in xlsx_files) else ""

def parse_birads(val):
    s = str(val).strip().upper().replace("BI-RADS", "").replace("BIRADS", "").replace(" ", "")
    nums = [int(c) for c in s if c.isdigit()]
    if not nums: return -1
    n = nums[0]
    return 0 if n == 0 else (1 if n in [1, 2] else (2 if n in [4, 5] else -1))

def read_excel_patients(xlsx_path):
    df = pd.read_excel(xlsx_path)
    patient_records = []
    birads_col = next((col for col in df.columns if any(x in str(col).upper() for x in ["BIRADS", "BI-RADS", "KATEGORİ"])), "BIRADS CATEGORY")
    case_col = next((col for col in df.columns if any(x in str(col).upper() for x in ["CASE", "HASTA", "ID", "NO"])), "CASENUMBER")
    density_col = next((col for col in df.columns if any(x in str(col).upper() for x in ["DENSITY", "YOĞUNLUK", "ACR"])), None)
    source_name = "train1" if "TRAIN1" in xlsx_path.upper() else "train2"

    seen_cases = set()
    for _, row in df.iterrows():
        case_num = row.get(case_col)
        if pd.isna(case_num): continue
        try: case_num = int(case_num)
        except: continue
        if case_num in seen_cases: continue

        label = parse_birads(row.get(birads_col))
        if label == -1: continue

        seen_cases.add(case_num)
        patient_records.append({
            "case": case_num,
            "label": label,
            "source": source_name,
            "density": str(row.get(density_col, "")) if density_col else ""
        })
    return patient_records

patient_recs = []
if TRAIN1: patient_recs += read_excel_patients(TRAIN1)
if USE_TRAIN2 and TRAIN2: patient_recs += read_excel_patients(TRAIN2)

df_patients = pd.DataFrame(patient_recs)
print(f"Toplam Tekil Hasta (Case) Sayısı: {len(df_patients)}")
print("Hasta Bazlı Sınıf Dağılımı:\n", df_patients["label"].value_counts().sort_index())

train_pat, temp_pat = train_test_split(df_patients, test_size=0.30, stratify=df_patients["label"], random_state=42)
val_pat, test_pat   = train_test_split(temp_pat, test_size=0.50, stratify=temp_pat["label"], random_state=42)

assert len(set(train_pat["case"]) & set(val_pat["case"])) == 0
assert len(set(train_pat["case"]) & set(test_pat["case"])) == 0
assert len(set(val_pat["case"]) & set(test_pat["case"])) == 0
print("Hasta bazlı %70/%15/%15 ayrım doğrulandı (Hasta bazlı ayrım).")

def expand_to_views(pat_df):
    rows = []
    for _, r in pat_df.iterrows():
        for view in ["RCC.dcm", "LCC.dcm", "RMLO.dcm", "LMLO.dcm"]:
            rows.append({
                "case": int(r["case"]),
                "view": view,
                "label": int(r["label"]),
                "source": r["source"]
            })
    return pd.DataFrame(rows)

train_df = expand_to_views(train_pat)
val_df   = expand_to_views(val_pat)
test_df  = expand_to_views(test_pat)

def oversample_train_df(df):
    max_size = df["label"].value_counts().max()
    balanced_list = []
    for label in sorted(df["label"].unique()):
        cls_df = df[df["label"] == label].copy()
        cls_df["aug_copy"] = 0
        if len(cls_df) < max_size:
            extra = cls_df.sample(max_size - len(cls_df), replace=True, random_state=42).copy()
            extra["aug_copy"] = np.arange(1, len(extra) + 1)
            cls_df = pd.concat([cls_df, extra], ignore_index=True)
        balanced_list.append(cls_df)
    return pd.concat(balanced_list).sample(frac=1, random_state=42).reset_index(drop=True)

train_df_balanced = oversample_train_df(train_df)
val_df["aug_copy"]  = 0
test_df["aug_copy"] = 0

case_destinations = {}
for split_name, split_df in [("train", train_df_balanced), ("val", val_df), ("test", test_df)]:
    for idx, row in split_df.iterrows():
        key = (int(row["case"]), row["view"].lower())
        view_stem = row["view"].split(".")[0].upper()
        arcname = f"{split_name}/{CLASS_NAMES[int(row['label'])]}/{split_name}_case{int(row['case']):06d}_{view_stem}_{idx:06d}.jpg"
        case_destinations.setdefault(key, []).append((
            split_name, idx, CLASS_NAMES[int(row["label"])], int(row["case"]), view_stem, int(row["aug_copy"]), arcname
        ))

active_sources = set(df_patients["source"].dropna().unique())
print(f"Görüntü Hedefleri | Train: {len(train_df_balanced)} | Val: {len(val_df)} | Test: {len(test_df)}")


In [ ]:
# ==============================================================================
# Bölüm 4: Çok Kanallı Tıbbi Görüntü Ön İşleme Pipeline'ı (3-Channel Contrast Representation)
# ==============================================================================
clahe_fine   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
clahe_detail = cv2.createCLAHE(clipLimit=3.5, tileGridSize=(16, 16))

def preprocess_mammogram_dicom(ds, view_stem="", aug_copy=0, target_size=512):
    """DICOM mamografi görüntüleri için çok kanallı tıbbi ön işleme ve standardizasyon fonksiyonu.

    İşlem Adımları:
        1. VOI LUT dönüşümü ve polarite standardizasyonu (MONOCHROME1 -> MONOCHROME2).
        2. [p1, p99.5] persentil pencereleme ve dinamik aralık normalizasyonu.
        3. Morfolojik bağlı bileşen analizi ile radyo-opak harf ve artefakt maskelemesi.
        4. Anatomik yön standardizasyonu (göğüs duvarının dikey eksene göre sola hizalanması).
        5. En-boy oranını koruyan dolgulu boyutlandırma (512x512 letterbox).
        6. Çoklu kontrast kodlama (R=Gamma, G=CLAHE Fine, B=CLAHE Detail + Unsharp Mask).
    """
    try:
        arr = apply_voi_lut(ds.pixel_array, ds).astype(np.float32)
    except Exception:
        arr = ds.pixel_array.astype(np.float32)

    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        arr = arr.max() - arr

    arr = arr - arr.min()
    fg_pixels = arr[arr > np.percentile(arr, 15)]
    if len(fg_pixels) > 100:
        p_low, p_high = np.percentile(fg_pixels, 1.0), np.percentile(fg_pixels, 99.5)
        if p_high > p_low:
            arr = np.clip(arr, 0, p_high)
            arr_u8 = (arr / p_high * 255.0).astype(np.uint8)
        else:
            arr_u8 = (arr / max(arr.max(), 1e-5) * 255.0).astype(np.uint8)
    else:
        arr_u8 = (arr / max(arr.max(), 1e-5) * 255.0).astype(np.uint8)

    h, w = arr_u8.shape
    if h > 20 and w > 20:
        arr_u8[:4, :] = 0; arr_u8[-4:, :] = 0
        arr_u8[:, :4] = 0; arr_u8[:, -4:] = 0

    # Morfolojik bağlı bileşen analizi (Connected Component Analysis) ile radyo-opak harf ve artefakt maskelemesi
    binary = (arr_u8 > 12).astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if num_labels > 1:
        largest_idx = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        breast_mask = (labels == largest_idx)
        arr_u8 = np.where(breast_mask, arr_u8, 0).astype(np.uint8)
        x, y, bw, bh, _ = stats[largest_idx]
        if bw > 32 and bh > 32:
            arr_u8 = arr_u8[y:y+bh, x:x+bw]

    # Anatomik yön standardizasyonu: Göğüs duvarının sol dikey eksene hizalanması
    col_mass = arr_u8.mean(axis=0)
    if len(col_mass) > 10:
        if col_mass[len(col_mass)//2:].sum() > col_mass[:len(col_mass)//2].sum() * 1.05:
            arr_u8 = np.fliplr(arr_u8)

    if aug_copy > 0:
        angle = ((aug_copy % 5) - 2) * 2.5
        ch, cw = arr_u8.shape
        M = cv2.getRotationMatrix2D((cw / 2, ch / 2), angle, 1.0)
        arr_u8 = cv2.warpAffine(arr_u8, M, (cw, ch), flags=cv2.INTER_LINEAR, borderValue=0)

    # En-boy oranını koruyan dolgulu boyutlandırma (Letterbox padding, 512x512)
    ch, cw = arr_u8.shape
    scale = target_size / max(ch, cw)
    nh, nw = max(1, int(round(ch * scale))), max(1, int(round(cw * scale)))
    resized = cv2.resize(arr_u8, (nw, nh), interpolation=cv2.INTER_AREA)

    canvas = np.zeros((target_size, target_size), dtype=np.uint8)
    y_off = (target_size - nh) // 2
    canvas[y_off:y_off+nh, 0:nw] = resized
    fg = canvas > 5

    # Çoklu kontrast temsili: R=Global Gamma (γ=0.85), G=Yerel Kontrast (CLAHE), B=Yüksek Frekans Detay (Unsharp Mask)
    ch_r = np.where(fg, np.power(canvas.astype(np.float32) / 255.0, 0.85) * 255.0, 0).astype(np.uint8)
    ch_g = np.where(fg, clahe_fine.apply(canvas), 0).astype(np.uint8)
    ch_b_base = clahe_detail.apply(canvas)
    blur = cv2.GaussianBlur(ch_b_base, (0, 0), sigmaX=1.5)
    ch_b = np.where(fg, cv2.addWeighted(ch_b_base, 1.4, blur, -0.4, 0), 0).astype(np.uint8)

    return Image.fromarray(np.stack([ch_r, ch_g, ch_b], axis=-1))


In [ ]:
# ==============================================================================
# Bölüm 5: Akış Tabanlı DICOM İndirme, İşleme ve Arşiv Kontrol Noktası Yönetimi
# ==============================================================================
import struct

print("Sağlık Bakanlığı Açık Veri Portalı taranıyor...")
res = requests.get(DATASET_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
soup = BeautifulSoup(res.text, 'html.parser')
links = []
for a in soup.find_all('a', href=True):
    href = a['href']
    if 'Download' in href or 'Document' in href:
        text = a.get_text(strip=True).replace(" İndirmek İçin Tıklayınız", "").replace(" click to download", "")
        url = BASE_URL + href if href.startswith("/") else href
        links.append({"name": text, "url": url})

groups = {}
for item in links:
    name = item["name"]
    if "Bilgi" in name: continue
    group_name = name.split(".zip")[0] + ".zip" if ".zip" in name else name
    groups.setdefault(group_name, []).append(item)

def find_last_valid_pk_tail(part_file_path, tail_window_bytes=48 * 1024 * 1024):
    """
    Bir split-zip parçasının sonundaki (varsa bir sonraki parçaya taşan)
    son geçerli PK\x03\x04 Local File Header konumunu bulur ve o noktadan
    parça sonuna kadar olan baytları döndürür (~2-25 MB).
    Böylece sonraki oturum önceki parçaları tekrar indirmeden doğrudan bu noktadan devam eder!
    """
    fsize = os.path.getsize(part_file_path)
    read_len = min(fsize, tail_window_bytes)
    with open(part_file_path, "rb") as f:
        f.seek(fsize - read_len)
        buf = f.read(read_len)

    pos = len(buf)
    while True:
        idx = buf.rfind(b"PK\x03\x04", 0, pos)
        if idx == -1:
            break
        if idx + 30 <= len(buf):
            ver, flag, method = struct.unpack("<HHH", buf[idx+4:idx+10])
            name_len, extra_len = struct.unpack("<HH", buf[idx+26:idx+30])
            if method in (0, 8) and 1 <= name_len <= 512 and idx + 30 + name_len <= len(buf):
                fname_bytes = buf[idx+30:idx+30+name_len]
                try:
                    fname_str = fname_bytes.decode("utf-8")
                    if ".dcm" in fname_str.lower() or "/" in fname_str:
                        return buf[idx:]
                except Exception:
                    pass
        pos = idx
    return b""

def download_item_curl(item, dest_dir):
    name, url = item["name"], item["url"]
    dest = os.path.join(dest_dir, name)
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return name, True
    time.sleep(random.uniform(1.5, 3.0))
    cmd = ["curl", "--retry", "5", "--retry-delay", "3", "-H", "User-Agent: Mozilla/5.0",
           "-L", "-f", "--connect-timeout", "30", "-m", "600", "-o", dest, url]
    try:
        result = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
        if result.returncode == 0 and os.path.exists(dest) and os.path.getsize(dest) > 0:
            return name, True
        if os.path.exists(dest): os.remove(dest)
        return name, False
    except Exception:
        if os.path.exists(dest): os.remove(dest)
        return name, False

session_limit_reached = False

def flush_batch_to_zip(batch_items):
    """Her FLUSH_EVERY görüntüde bir zip dosyasını açıp yazar ve hemen kapatır (Central Directory korunur)."""
    if not batch_items:
        return
    with zipfile.ZipFile(out_zip_path, 'a', compression=zipfile.ZIP_DEFLATED) as zf_out:
        for arcname, img_bytes in batch_items:
            if arcname not in existing_arcnames:
                zf_out.writestr(arcname, img_bytes)
                existing_arcnames.add(arcname)
    batch_items.clear()

for g_name, g_items in groups.items():
    if session_limit_reached:
        break
    is_needed = ("EGITIM_1" in g_name and "train1" in active_sources) or ("EGITIM_2" in g_name and "train2" in active_sources)
    if not is_needed:
        continue
    if g_name in prep_state.get("completed_groups", []):
        print(f"{g_name} önceki oturumda tamamen tamamlanmış, atlanıyor!")
        continue

    sorted_items = sorted(g_items, key=lambda x: x["name"])
    g_prog = prep_state["group_progress"].get(g_name, {"last_completed_part_idx": -1, "saved_images": 0})
    resume_part_idx = int(g_prog.get("last_completed_part_idx", -1)) + 1

    print(f"\n=== {g_name} İşleniyor (Parça {resume_part_idx + 1}/{len(sorted_items)} -> {len(sorted_items)}) ===")
    write_buffer = []
    ok_count = int(g_prog.get("saved_images", 0))
    skip_count = 0

    def part_stream_generator():
        global session_limit_reached
        # Arşiv sınırında bölünen DICOM veri akışının bütünlüğünün sağlanması (carryover buffer)
        if resume_part_idx > 0 and os.path.exists(tail_bin_path):
            print(f"Parça {resume_part_idx} sınırından kalan {os.path.getsize(tail_bin_path)/1024/1024:.2f} MB carryover akışa eklendi.")
            with open(tail_bin_path, "rb") as tf:
                while True:
                    c = tf.read(262144)
                    if not c: break
                    yield c

        for idx in range(resume_part_idx, len(sorted_items)):
            if (time.time() - SESSION_START_TIME) / 3600.0 >= MAX_SESSION_HOURS:
                print(f"\n Oturum süresi sınırı ({MAX_SESSION_HOURS} saat) doldu! Mevcut parça sonunda güvenle duruluyor...")
                session_limit_reached = True
                break

            item = sorted_items[idx]
            name = item["name"]
            dest = os.path.join(TEMP_DIR, name)
            success = False
            for attempt in range(1, 6):
                _, ok = download_item_curl(item, TEMP_DIR)
                if ok:
                    success = True
                    break
                print(f"{name} indirilemedi, tekrar deneniyor ({attempt}/5)...")
                time.sleep(10 * attempt)
            if not success:
                raise RuntimeError(f"Kritik Hata: {name} indirilemedi!")

            # Kesintiye uğrayan indirme süreçlerinde parça doğrulaması için ZIP başlık kuyruğunun kaydedilmesi
            next_tail = find_last_valid_pk_tail(dest)

            with open(dest, "rb") as pf:
                while True:
                    chunk = pf.read(262144)
                    if not chunk: break
                    yield chunk

            try: os.remove(dest)
            except: pass

            # İşlenen arşiv parçası sonrası bellek tamponunun diske yazılması ve kontrol noktası kaydı
            flush_batch_to_zip(write_buffer)
            if next_tail:
                with open(tail_bin_path + ".tmp", "wb") as tf:
                    tf.write(next_tail)
                os.replace(tail_bin_path + ".tmp", tail_bin_path)
            prep_state["group_progress"][g_name] = {
                "last_completed_part_idx": idx,
                "last_completed_part_name": name,
                "saved_images": len(existing_arcnames)
            }
            save_prep_state()
            print(f"[{idx+1}/{len(sorted_items)}] {name} kaydedildi (Toplam JPEG: {len(existing_arcnames)}).")

    try:
        for file_name_bytes, file_size, unzipped_chunks in stream_unzip(part_stream_generator()):
            file_name = file_name_bytes.decode('utf-8', errors='ignore')
            if not file_name.lower().endswith(".dcm"):
                for _ in unzipped_chunks: pass
                continue
            parts = file_name.strip("/").split("/")
            if len(parts) < 2:
                for _ in unzipped_chunks: pass
                continue
            try:
                casenumber = int(parts[-2])
            except Exception:
                for _ in unzipped_chunks: pass
                continue

            filename = os.path.basename(file_name).lower()
            dest_info = case_destinations.get((casenumber, filename))
            if not dest_info:
                for _ in unzipped_chunks: pass
                continue

            pending_dest = [d for d in dest_info if d[6] not in existing_arcnames]
            if not pending_dest:
                for _ in unzipped_chunks: pass
                continue

            dcm_data = b"".join(unzipped_chunks)
            try:
                ds = pydicom.dcmread(io.BytesIO(dcm_data))
                for split, idx, label_name, case_id, view_stem, aug_copy, arcname in pending_dest:
                    img = preprocess_mammogram_dicom(ds, view_stem=view_stem, aug_copy=aug_copy, target_size=IMG_SIZE)
                    buf = io.BytesIO()
                    img.save(buf, format="JPEG", quality=94)
                    write_buffer.append((arcname, buf.getvalue()))
                    ok_count += 1
                if len(write_buffer) >= FLUSH_EVERY:
                    flush_batch_to_zip(write_buffer)
            except Exception:
                skip_count += 1
    except Exception as stream_err:
        # Arşiv merkezi dizin yapısının doğrulanması ve son tamponun diske aktarımı
        flush_batch_to_zip(write_buffer)
        print(f"ℹ Akış sonlandı ({stream_err}). Tampon güvenle diske kaydedildi.")

    flush_batch_to_zip(write_buffer)
    if not session_limit_reached:
        if g_name not in prep_state["completed_groups"]:
            prep_state["completed_groups"].append(g_name)
        save_prep_state()
        print(f"{g_name} tamamlandı. Toplam kayıtlı JPEG: {len(existing_arcnames)}")
    else:
        save_prep_state()
        print("Oturum süresi sınırı nedeniyle işlem durduruldu ve checkpoint kaydedildi.")


In [ ]:
# ==============================================================================
# Bölüm 6: Veri Seti Bütünlüğü Doğrulaması ve Sınıf Dağılım İstatistikleri
# ==============================================================================
print("\n--- Veri Seti İstatistikleri ---")
if os.path.exists(out_zip_path):
    with zipfile.ZipFile(out_zip_path, 'r') as ref:
        namelist = ref.namelist()
        for split in ["train", "val", "test"]:
            total = 0
            print(f"{split.upper()} Klasörü:")
            for cls in CLASS_NAMES:
                prefix = f"{split}/{cls}/"
                cnt = sum(1 for name in namelist if name.startswith(prefix) and name.endswith(".jpg"))
                total += cnt
                print(f"- {cls}: {cnt} görüntü")
            print(f"Toplam: {total} görüntü\n")
    print(f"yolo_dataset.zip oluşturuldu ve doğrulandı. ({out_zip_path})")
